In [ ]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox, simpledialog
from tkinter.colorchooser import askcolor
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
import dlisio
import pandas as pd
import numpy as np
import os
import copy
import json
import traceback
import re

# =============================================================================
# 1. CONFIGURAÇÕES E CONSTANTES
# =============================================================================
CONFIG_LIB_FILENAME = "user_library.json"
CONFIG_STYLE_FILENAME = "user_style.json"

DEFAULT_MAP = {
    'TIME':     ['TIME', 'ETIM', 'NUM', 'T', 'SECONDS', 'Time', 'E-Time'],
    'PRESSURE': ['BQP1', 'BRQ1', 'BPC1', 'QPR', 'P_QUARTZ', 'CQ',  
                 'BSG1', 'BRS1', 'SPRE', 'STRAIN', 'HP', 'P1', 'PRESS', 'PA', 'PS', 'PRE', 'FPRE'],                                  
    'VOLUME':   ['PTV1', 'VPRE', 'CHVOL', 'VOL', 'LVOL', 'SVOL', 'HPV1', 'POPV', 'PISTON', 'V_CH1', 'SC1', 'SC2'],
    'DEPTH':    ['DVP1', 'RDVP1', 'TDEP', 'DEPT', 'DEPTH', 'MD', 'DEPT_COR']
}

DEFAULT_STYLE = {
    'col_pres_raw': '#FF7F7F',    'lw_pres_raw': 1.0,    'ls_pres_raw': '-',    'alpha_pres_raw': 0.3,
    'col_pres_smooth': '#8B0000', 'lw_pres_smooth': 1.5, 'ls_pres_smooth': '-', 'alpha_pres_smooth': 1.0,
    'col_vol': '#1f77b4',         'lw_vol': 1.5,         'ls_vol': '-',         'alpha_vol': 1.0,
    'label_x': 'Tempo',
    'label_y_left': 'Pressão (psi)',
    'label_y_right': 'Volume (cm3)',
    'title_prefix': '',          
    'show_grid': True, 'grid_color': '#b0b0b0', 'grid_alpha': 0.5, 'grid_linewidth': 0.8,
    'fs_title': 12, 'fs_label_x': 10, 'fs_label_y': 10,
    'invert_y_left': False, 'invert_y_right': False,
    'legend_loc': 'upper right',
    'plot_bgcolor': 'white',
    'vol_plot_bgcolor': 'white', 
    'convert_psi_kgf': False, 
    'conversion_factor': 14.2233, 
    'conversion_label': 'kgf/cm²',
    'use_time_conv': False, 
    'time_conv_factor': 60.0, 
    'show_smooth': True,
    'tick_step_x': 0.0,
    'tick_step_y': 0.0,
    'tick_step_y_right': 0.0
}

LEGEND_LOC_MAP = {"Direita Superior": "upper right", "Direita Inferior": "lower right", "Esquerda Superior": "upper left", "Esquerda Inferior": "lower left", "Melhor Ajuste": "best"}
LEGEND_LOC_MAP_REV = {v: k for k, v in LEGEND_LOC_MAP.items()}

PSI_TO_KGF = 14.2233

# --- UTILITÁRIOS ---
def center_window_on_parent(window, parent):
    window.update_idletasks()
    w = window.winfo_reqwidth(); h = window.winfo_reqheight()
    p_w = parent.winfo_width(); p_h = parent.winfo_height()
    p_x = parent.winfo_rootx(); p_y = parent.winfo_rooty()
    x = p_x + (p_w // 2) - (w // 2); y = p_y + (p_h // 2) - (h // 2)
    window.geometry(f'+{x}+{y}')

def force_fit_to_screen(window):
    window.update_idletasks()
    req_w = window.winfo_reqwidth(); req_h = window.winfo_reqheight()
    screen_w = window.winfo_screenwidth(); screen_h = window.winfo_screenheight()
    final_w = min(req_w, screen_w - 20); final_h = min(req_h, screen_h - 80)
    pos_x = (screen_w // 2) - (final_w // 2); pos_y = (screen_h // 2) - (final_h // 2)
    window.geometry(f"{final_w}x{final_h}+{pos_x}+{pos_y}")

def load_library_from_disk():
    if os.path.exists(CONFIG_LIB_FILENAME):
        try:
            with open(CONFIG_LIB_FILENAME, 'r') as f:
                data = json.load(f); 
                if 'FLUID' in data: del data['FLUID']
                return data
        except: pass
    return copy.deepcopy(DEFAULT_MAP)

def save_json(filename, data):
    try:
        with open(filename, 'w') as f: json.dump(data, f, indent=4)
        return True
    except: return False

# =============================================================================
# 2. BACKEND
# =============================================================================
def extract_well_name(logical_file):
    try:
        for origin in logical_file.origins:
            if hasattr(origin, 'well_name') and origin.well_name: return origin.well_name.strip()
    except: pass
    try:
        for param in logical_file.parameters:
            if param.name.upper() in ['WN', 'WELL'] and len(param.values) > 0: return str(param.values[0]).strip()
    except: pass
    return "POÇO DESCONHECIDO"

def detect_tool_and_type(pressure_channel_name, all_frame_channels):
    p_name = pressure_channel_name.upper()
    chans = [c.upper() for c in all_frame_channels]
    if p_name in ['BQP1', 'BRQ1', 'BPC1', 'QPR', 'P_QUARTZ', 'CQ']: return "MDT-Quartz"
    if p_name in ['BSG1', 'BRS1']: return "MDT-Strain"
    if p_name in ['SPRE', 'STRAIN', 'HP', 'P1', 'PA', 'PS', 'PRESS']: return "SFT"
    if 'PTV1' in chans or 'DVP1' in chans or 'OILF_LFA' in chans: return "MDT"
    if any(x in chans for x in ['VPRE', 'CHVOL', 'SC1', 'SC2']): return "SFT"
    return "TEST"

def get_safe_index_name(frame):
    if hasattr(frame.index, 'name'): return frame.index.name
    elif isinstance(frame.index, str): return frame.index
    return "UNKNOWN"

def get_all_channels_from_dlis(logical_files):
    all_chans = set()
    for lf in logical_files:
        for frame in lf.frames:
            for c in frame.channels: all_chans.add(c.name)
            idx = get_safe_index_name(frame)
            if idx != "UNKNOWN": all_chans.add(idx)
    return sorted(list(all_chans))

def guess_channels(available_channels, current_map):
    guesses = {}
    for cat, priorities in current_map.items():
        for cand in priorities:
            if cand in available_channels: guesses[cat] = cand; break
    return guesses

def extract_data_with_map(logical_files, active_map):
    dfs = {}; count = 0
    for i, lf in enumerate(logical_files):
        for frame in lf.frames:
            chans = [c.name for c in frame.channels]; idx_name = get_safe_index_name(frame)
            if any(x in chans for x in active_map['PRESSURE']):
                try:
                    curves = frame.curves(); data = {}; used_meta = {}
                    for cat, cands in active_map.items():
                        found = False
                        for cand in cands:
                            if cand in chans:
                                try:
                                    v = curves[cand]
                                    if hasattr(v, 'ndim') and v.ndim > 1: v = v.flatten()
                                    data[cat] = v; used_meta[cat] = cand; found = True; break
                                except: continue
                            elif cat == 'TIME' and idx_name == cand:
                                v = curves[frame.index] if (isinstance(frame.index, str) and frame.index in curves.dtype.names) else frame.index
                                if hasattr(v, 'ndim') and v.ndim > 1: v = v.flatten()
                                data['TIME'] = v; used_meta['TIME'] = f"{cand} (Idx)"; found = True; break
                        if cat == 'TIME' and not found:
                             try:
                                 v = frame.index
                                 if hasattr(v, 'ndim') and v.ndim > 1: v = v.flatten()
                                 if len(v) > 10 and v[-1] > v[0]:
                                     data['TIME'] = v; used_meta['TIME'] = "Index (Auto)"; found = True
                             except: pass

                    if 'PRESSURE' in data:
                        l_ref = len(data['PRESSURE'])
                        valid_data = {k:v for k,v in data.items() if len(v) == l_ref}
                        if 'PRESSURE' in valid_data:
                            df = pd.DataFrame(valid_data)
                            df.replace(-999.25, np.nan, inplace=True)
                            if df['PRESSURE'].isna().all(): continue
                            df['PRE_SMOOTH'] = df['PRESSURE'].rolling(window=15, min_periods=1, center=True).mean()
                            d_val = df['DEPTH'].median() if 'DEPTH' in df.columns else 0.0
                            d_lbl = f"@{d_val:.1f}m"
                            p_nm = used_meta.get('PRESSURE', 'Unk')
                            tool = detect_tool_and_type(p_nm, chans)
                            count += 1
                            key = f"[{tool}] ({p_nm}) #{count:02d} | {d_lbl}"
                            df.attrs['tool'] = tool; df.attrs['channels_used'] = used_meta
                            dfs[key] = df
                except Exception as e: 
                    print(f"Erro no frame: {e}"); pass
    return dfs

def backend_load_dlis_smart(filepath, active_map, root_window_for_dialog=None):
    try:
        try: raw = dlisio.load(filepath)
        except: raw = dlisio.dlis.load(filepath)
    except Exception as e: return None, None, None, f"Falha DLIS: {str(e)}"
    lfs = []
    if isinstance(raw, (list, tuple)): lfs = [o for o in raw if hasattr(o, 'frames')]
    elif hasattr(raw, 'frames'): lfs = [raw]
    if not lfs: return None, None, None, "Arquivo DLIS sem estrutura lógica válida."
    wname = extract_well_name(lfs[0])
    res = extract_data_with_map(lfs, active_map)
    if res: return res, lfs, wname, None
    if root_window_for_dialog:
        all_c = get_all_channels_from_dlis(lfs)
        if not all_c: return None, None, wname, "Arquivo vazio."
        guess = guess_channels(all_c, active_map)
        dial = ChannelMapperDialog(root_window_for_dialog, all_c, guess)
        if dial.result: return extract_data_with_map(lfs, dial.result), lfs, wname, None
    return None, None, wname, "Nenhum dado compatível encontrado."

# =============================================================================
# 3. INTERFACE
# =============================================================================
class ExportOptionWindow(tk.Toplevel):
    def __init__(self, parent):
        super().__init__(parent)
        self.parent_app = parent
        self.title("Exportar")
        self.geometry("300x250")
        self.v = tk.StringVar(value="LAS2")
        ttk.Label(self, text="Formato:", font=("Arial", 10, "bold")).pack(pady=15)
        frm = ttk.Frame(self); frm.pack(fill='x', padx=40)
        ttk.Radiobutton(frm, text="LAS 2.0 (Padrão)", variable=self.v, value="LAS2").pack(anchor='w', pady=5)
        ttk.Radiobutton(frm, text="LAS 3.0 (Moderno)", variable=self.v, value="LAS3").pack(anchor='w', pady=5)
        ttk.Radiobutton(frm, text="CSV (Excel)", variable=self.v, value="CSV").pack(anchor='w', pady=5)
        ttk.Button(self, text="Continuar", command=self.confirm).pack(pady=20)
        self.transient(parent); center_window_on_parent(self, parent); self.grab_set()
    def confirm(self):
        val = self.v.get(); c = "LAS" if "LAS" in val else "CSV"; v = "2.0" if val=="LAS2" else ("3.0" if val=="LAS3" else None)
        self.destroy(); self.parent_app.perform_export(c, v)

class SelectEventDialog(tk.Toplevel):
    def __init__(self, parent, events_list, on_select_callback):
        super().__init__(parent)
        self.title("Mover Evento")
        self.geometry("300x150")
        self.on_select = on_select_callback
        ttk.Label(self, text="Selecione o evento para mover:", padding=10).pack()
        self.cb = ttk.Combobox(self, values=events_list, state="readonly")
        self.cb.pack(padx=20, pady=5, fill='x')
        if events_list: self.cb.current(0)
        ttk.Button(self, text="Confirmar", command=self.confirm).pack(pady=10)
        center_window_on_parent(self, parent); self.grab_set()
    def confirm(self):
        val = self.cb.get()
        if val: self.on_select(val); self.destroy()

class StyleEditorWindow(tk.Toplevel):
    def __init__(self, parent, current_style, on_apply_callback):
        super().__init__(parent)
        self.title("Editor de Estilo")
        self.style = copy.deepcopy(current_style)
        self.on_apply = on_apply_callback
        P_IN=5; P_OUT=5
        self.frame_btns = ttk.Frame(self, padding=10)
        self.frame_btns.pack(side='bottom', fill='x')
        ttk.Button(self.frame_btns, text="Restaurar Padrão", command=self.reset_defaults).pack(side='left')
        ttk.Button(self.frame_btns, text="✅ Aplicar", command=self.apply).pack(side='right')
        self.frame_files = ttk.LabelFrame(self, text="Gerenciar Estilo (JSON)", padding=P_IN)
        self.frame_files.pack(fill='x', padx=P_OUT, pady=2, side='top')
        ttk.Button(self.frame_files, text="📂 Importar", command=self.import_json).pack(side='left', expand=True, fill='x', padx=2)
        ttk.Button(self.frame_files, text="💾 Exportar", command=self.export_json).pack(side='right', expand=True, fill='x', padx=2)
        self.nb = ttk.Notebook(self)
        self.nb.pack(fill='both', expand=True, padx=P_OUT, pady=5)
        self.tab_curves = ttk.Frame(self.nb, padding=P_IN); self.tab_layout = ttk.Frame(self.nb, padding=P_IN)
        self.tab_grid = ttk.Frame(self.nb, padding=P_IN); self.tab_text = ttk.Frame(self.nb, padding=P_IN)
        self.nb.add(self.tab_curves, text="Curvas"); self.nb.add(self.tab_layout, text="Layout")
        self.nb.add(self.tab_grid, text="Grid"); self.nb.add(self.tab_text, text="Textos")
        self.btn_colors = {}; self.spin_lw = {}; self.combo_ls = {}; self.scale_alpha = {}
        self.var_show_smooth = tk.BooleanVar(value=self.style.get('show_smooth', True))
        ttk.Checkbutton(self.tab_curves, text="Exibir Curva Suavizada", variable=self.var_show_smooth).pack(anchor='w', pady=(0,5))
        self.create_curve_row(self.tab_curves, "Pressão Bruta:", 'col_pres_raw', 'lw_pres_raw', 'ls_pres_raw', 'alpha_pres_raw')
        self.create_curve_row(self.tab_curves, "Pressão Suavizada:", 'col_pres_smooth', 'lw_pres_smooth', 'ls_pres_smooth', 'alpha_pres_smooth')
        self.create_curve_row(self.tab_curves, "Volume:", 'col_vol', 'lw_vol', 'ls_vol', 'alpha_vol')
        self.entries = {} 
        self.var_inv_left = tk.BooleanVar(value=self.style.get('invert_y_left', False))
        self.var_inv_right = tk.BooleanVar(value=self.style.get('invert_y_right', False))
        ttk.Checkbutton(self.tab_layout, text="Inverter Pressão (Esq)", variable=self.var_inv_left).pack(anchor='w', pady=2)
        ttk.Checkbutton(self.tab_layout, text="Inverter Volume (Dir)", variable=self.var_inv_right).pack(anchor='w', pady=2)
        ttk.Separator(self.tab_layout, orient='horizontal').pack(fill='x', pady=5)
        lf_ticks = ttk.LabelFrame(self.tab_layout, text="Intervalo de Ticks (0 = Automático)", padding=5)
        lf_ticks.pack(fill='x', pady=2)
        self.create_text_row(lf_ticks, "Tick X (Passo):", 'tick_step_x')
        self.create_text_row(lf_ticks, "Tick Y Esq (Passo):", 'tick_step_y')
        self.create_text_row(lf_ticks, "Tick Y Dir (Passo):", 'tick_step_y_right')
        ttk.Separator(self.tab_layout, orient='horizontal').pack(fill='x', pady=5)
        self.var_use_conv = tk.BooleanVar(value=self.style.get('convert_psi_kgf', False))
        f_conv = ttk.Frame(self.tab_layout); f_conv.pack(fill='x', pady=2)
        ttk.Checkbutton(f_conv, text="Ativar Conversão de Pressão", variable=self.var_use_conv).pack(side='left')
        self.cb_conv_mode = ttk.Combobox(self.tab_layout, values=["PSI -> kgf/cm² (÷ 14.22)", "kgf/cm² -> PSI (x 14.22)"], state="readonly", width=30)
        curr_factor = self.style.get('conversion_factor', 14.2233)
        if curr_factor < 1.0: self.cb_conv_mode.current(1)
        else: self.cb_conv_mode.current(0)
        self.cb_conv_mode.pack(anchor='w', padx=25)
        self.var_use_time_conv = tk.BooleanVar(value=self.style.get('use_time_conv', False))
        f_tconv = ttk.Frame(self.tab_layout); f_tconv.pack(fill='x', pady=2)
        ttk.Checkbutton(f_tconv, text="Converter Tempo (Divisor):", variable=self.var_use_time_conv).pack(side='left')
        self.ent_time_factor = ttk.Entry(self.tab_layout, width=10)
        self.ent_time_factor.insert(0, str(self.style.get('time_conv_factor', 60.0))); self.ent_time_factor.pack(anchor='w', padx=25)
        ttk.Separator(self.tab_layout, orient='horizontal').pack(fill='x', pady=5)
        f_leg = ttk.Frame(self.tab_layout); f_leg.pack(fill='x', pady=5)
        ttk.Label(f_leg, text="Legenda:", width=20).pack(side='left')
        self.cb_legend = ttk.Combobox(f_leg, values=list(LEGEND_LOC_MAP.keys()), state="readonly")
        self.cb_legend.set(LEGEND_LOC_MAP_REV.get(self.style.get('legend_loc', 'upper right'), "Direita Superior"))
        self.cb_legend.pack(side='left', fill='x', expand=True)
        self.var_show_grid = tk.BooleanVar(value=self.style.get('show_grid', True))
        ttk.Checkbutton(self.tab_grid, text="Exibir Grid no Gráfico", variable=self.var_show_grid).pack(anchor='w', pady=(0,10))
        f_g1 = ttk.Frame(self.tab_grid); f_g1.pack(fill='x', pady=2)
        self.btn_grid_col = tk.Button(f_g1, text="   ", bg=self.style.get('grid_color', '#b0b0b0'), width=6, command=lambda: self.pick_color_direct('grid_color', self.btn_grid_col))
        self.btn_grid_col.pack(side='left'); self.btn_colors['grid_color'] = self.btn_grid_col
        f_bg = ttk.Frame(self.tab_grid); f_bg.pack(fill='x', pady=2)
        self.btn_bg_col = tk.Button(f_bg, text="   ", bg=self.style.get('plot_bgcolor', 'white'), width=6, command=lambda: self.pick_color_direct('plot_bgcolor', self.btn_bg_col))
        self.btn_bg_col.pack(side='left'); self.btn_colors['plot_bgcolor'] = self.btn_bg_col

        # NOVO: Fundo do Volume
        f_bg_vol = ttk.Frame(self.tab_grid); f_bg_vol.pack(fill='x', pady=2)
        ttk.Label(f_bg_vol, text="Cor Fundo (Vol):", width=15).pack(side='left')
        self.btn_bg_vol_col = tk.Button(f_bg_vol, text="   ", bg=self.style.get('vol_plot_bgcolor', 'white'), width=6, command=lambda: self.pick_color_direct('vol_plot_bgcolor', self.btn_bg_vol_col))
        self.btn_bg_vol_col.pack(side='left'); self.btn_colors['vol_plot_bgcolor'] = self.btn_bg_vol_col

        f_g2 = ttk.Frame(self.tab_grid); f_g2.pack(fill='x', pady=5)
        self.scale_grid_alpha = tk.Scale(f_g2, from_=0.1, to=1.0, resolution=0.1, orient='horizontal')
        self.scale_grid_alpha.set(self.style.get('grid_alpha', 0.5)); self.scale_grid_alpha.pack(side='left', fill='x', expand=True)
        f_g3 = ttk.Frame(self.tab_grid); f_g3.pack(fill='x', pady=5)
        self.spin_grid_lw = ttk.Spinbox(f_g3, from_=0.5, to=3.0, increment=0.5, width=6)
        self.spin_grid_lw.set(self.style.get('grid_linewidth', 0.8)); self.spin_grid_lw.pack(side='left')
        self.spin_fonts = {}
        lf_f = ttk.LabelFrame(self.tab_text, text="Fontes", padding=5); lf_f.pack(fill='x')
        self.create_font_row(lf_f, "Título:", 'fs_title'); self.create_font_row(lf_f, "Eixo X:", 'fs_label_x'); self.create_font_row(lf_f, "Eixos Y:", 'fs_label_y')
        lf_t = ttk.LabelFrame(self.tab_text, text="Rótulos", padding=5); lf_t.pack(fill='x', pady=5)
        self.create_text_row(lf_t, "Eixo X:", 'label_x'); self.create_text_row(lf_t, "Eixo Y (Esq):", 'label_y_left'); self.create_text_row(lf_t, "Eixo Y (Dir):", 'label_y_right'); self.create_text_row(lf_t, "Prefixo:", 'title_prefix')
        self.transient(parent); force_fit_to_screen(self); self.grab_set()

    def create_curve_row(self, parent, label, key_col, key_lw, key_ls, key_alpha):
        f = ttk.Frame(parent); f.pack(fill='x', pady=2)
        ttk.Label(f, text=label, width=18).pack(side='left')
        s_alpha = tk.Scale(f, from_=0.0, to=1.0, resolution=0.1, orient='horizontal', length=50, showvalue=0)
        s_alpha.set(self.style.get(key_alpha, 1.0)); s_alpha.pack(side='right', padx=2); self.scale_alpha[key_alpha] = s_alpha
        cb = ttk.Combobox(f, values=['-', '--', ':', '-.'], width=3, state="readonly")
        cb.set(self.style.get(key_ls, '-')); cb.pack(side='right', padx=2); self.combo_ls[key_ls] = cb
        spin = ttk.Spinbox(f, from_=0.5, to=5.0, increment=0.5, width=4)
        spin.set(self.style.get(key_lw, 1.0)); spin.pack(side='right', padx=2); self.spin_lw[key_lw] = spin
        btn = tk.Button(f, text="  ", bg=self.style[key_col], width=4, command=lambda k=key_col: self.pick_color(k))
        btn.pack(side='right', padx=2); self.btn_colors[key_col] = btn
    def create_font_row(self, parent, label, key_fs):
        f = ttk.Frame(parent); f.pack(fill='x', pady=2); ttk.Label(f, text=label, width=15).pack(side='left')
        spin = ttk.Spinbox(f, from_=6, to=24, increment=1, width=5); spin.set(self.style.get(key_fs, 10)); spin.pack(side='right', padx=5); self.spin_fonts[key_fs] = spin
    def create_text_row(self, parent, label, key):
        f = ttk.Frame(parent); f.pack(fill='x', pady=2); ttk.Label(f, text=label, width=15).pack(side='left')
        ent = ttk.Entry(f); ent.pack(side='right', fill='x', expand=True); ent.insert(0, str(self.style.get(key, ''))); self.entries[key] = ent
    def pick_color(self, key):
        c = askcolor(color=self.style[key], title="Cor")[1]
        if c: self.style[key] = c; self.btn_colors[key].config(bg=c)
    def pick_color_direct(self, key, btn_widget):
        c = askcolor(color=self.style.get(key, '#000000'), title="Cor")[1]
        if c: self.style[key] = c; btn_widget.config(bg=c)
    def reset_defaults(self):
        if messagebox.askyesno("Reset", "Voltar ao padrão?"): self.style = copy.deepcopy(DEFAULT_STYLE); self.destroy(); self.on_apply(self.style)
    def import_json(self):
        path = filedialog.askopenfilename(filetypes=[("JSON", "*.json")])
        if path:
            try:
                with open(path, 'r') as f: self.style.update(json.load(f)); self.apply()
            except: pass
    def export_json(self):
        self._capture_ui_values(); path = filedialog.asksaveasfilename(defaultextension=".json", filetypes=[("JSON", "*.json")])
        if path: save_json(path, self.style)
    def _capture_ui_values(self):
        for key, ent in self.entries.items(): self.style[key] = ent.get()
        try: self.style['tick_step_x'] = float(self.entries['tick_step_x'].get())
        except: self.style['tick_step_x'] = 0.0
        try: self.style['tick_step_y'] = float(self.entries['tick_step_y'].get())
        except: self.style['tick_step_y'] = 0.0
        try: self.style['tick_step_y_right'] = float(self.entries['tick_step_y_right'].get())
        except: self.style['tick_step_y_right'] = 0.0
        for key, spin in self.spin_lw.items(): 
            try: self.style[key] = float(spin.get())
            except: pass
        for key, cb in self.combo_ls.items(): self.style[key] = cb.get()
        for key, scale in self.scale_alpha.items(): self.style[key] = scale.get()
        for key, spin in self.spin_fonts.items():
            try: self.style[key] = int(spin.get())
            except: pass
        self.style['invert_y_left'] = self.var_inv_left.get(); self.style['invert_y_right'] = self.var_inv_right.get()
        self.style['show_grid'] = self.var_show_grid.get(); self.style['grid_alpha'] = self.scale_grid_alpha.get()
        try: self.style['grid_linewidth'] = float(self.spin_grid_lw.get())
        except: pass
        self.style['legend_loc'] = LEGEND_LOC_MAP.get(self.cb_legend.get(), 'upper right')
        self.style['convert_psi_kgf'] = self.var_use_conv.get()
        if self.cb_conv_mode.current() == 0: 
            self.style['conversion_factor'] = PSI_TO_KGF
            self.style['conversion_label'] = 'kgf/cm²'
        else: 
            self.style['conversion_factor'] = 1.0 / PSI_TO_KGF
            self.style['conversion_label'] = 'PSI'
        self.style['show_smooth'] = self.var_show_smooth.get(); self.style['use_time_conv'] = self.var_use_time_conv.get()
        try: self.style['time_conv_factor'] = float(self.ent_time_factor.get())
        except: pass
    def apply(self): self._capture_ui_values(); self.on_apply(self.style); self.destroy()

class CategorySelectorWindow(tk.Toplevel):
    def __init__(self, parent, current_map, on_save_callback):
        super().__init__(parent)
        self.title("Biblioteca"); self.geometry("350x400")
        self.current_map = current_map; self.on_save_callback = on_save_callback
        ttk.Label(self, text="Categoria:", font=("Arial", 11, "bold"), padding=15).pack()
        frm_btns = ttk.Frame(self, padding=20); frm_btns.pack(fill='both', expand=True)
        icons = {'PRESSURE': '🔴', 'TIME': '⏱️', 'VOLUME': '💧', 'DEPTH': '📏'}
        for cat in self.current_map.keys():
            btn = ttk.Button(frm_btns, text=f"{icons.get(cat, '📁')}  {cat}", command=lambda c=cat: MnemonicEditorWindow(self, c, self.current_map, self.on_save_callback))
            btn.pack(fill='x', pady=5, ipady=5)
        ttk.Button(self, text="Restaurar Padrão", command=self.reset_defaults).pack(fill='x', padx=20, pady=5)
        self.transient(parent); center_window_on_parent(self, parent); self.grab_set()
    def reset_defaults(self):
        if messagebox.askyesno("Reset", "Restaurar biblioteca?"):
            self.current_map.clear(); self.current_map.update(copy.deepcopy(DEFAULT_MAP)); save_json(CONFIG_LIB_FILENAME, self.current_map); messagebox.showinfo("Ok", "Restaurado.")

class MnemonicEditorWindow(tk.Toplevel):
    def __init__(self, parent, category, full_map, on_save_callback):
        super().__init__(parent)
        self.title(f"Editando: {category}"); self.geometry("400x500")
        self.category = category; self.mnemonics_list = full_map[category]; self.full_map = full_map; self.save_cb = on_save_callback
        self.app_ref = parent.master
        self.available_channels = None
        if hasattr(self.app_ref, 'logical_files_store') and self.app_ref.logical_files_store: self.available_channels = set(get_all_channels_from_dlis(self.app_ref.logical_files_store))
        ttk.Label(self, text=f"Lista: {category}", font=("Arial", 10, "bold"), padding=10).pack()
        ttk.Label(self, text="⬛ Padrão de Fábrica   🟥 Personalizado", font=("Arial", 8)).pack(pady=(0,5))
        frm_list = ttk.Frame(self, padding=10); frm_list.pack(fill='both', expand=True)
        self.lb = tk.Listbox(frm_list, font=("Consolas", 10)); sb = ttk.Scrollbar(frm_list, command=self.lb.yview); self.lb.config(yscrollcommand=sb.set)
        sb.pack(side='right', fill='y'); self.lb.pack(side='left', fill='both', expand=True)
        frm_in = ttk.Frame(self, padding=10); frm_in.pack(fill='x')
        self.ent = ttk.Entry(frm_in); self.ent.pack(side='left', fill='x', expand=True)
        self.ent.bind('<Return>', lambda e: self.add()); ttk.Button(frm_in, text="Add", command=self.add).pack(side='left')
        frm_bt = ttk.Frame(self, padding=10); frm_bt.pack(fill='x')
        ttk.Button(frm_bt, text="Remover", command=self.rem).pack(side='left'); ttk.Button(frm_bt, text="Salvar", command=self.save).pack(side='right')
        self.refresh(); self.transient(parent); center_window_on_parent(self, parent); self.grab_set()
    def refresh(self):
        self.lb.delete(0, tk.END); defaults = DEFAULT_MAP.get(self.category, [])
        for i, m in enumerate(self.mnemonics_list):
            self.lb.insert(tk.END, m); self.lb.itemconfig(i, {'fg': 'black' if m in defaults else 'red'})
    def add(self):
        v = self.ent.get().strip().upper()
        if v and v not in self.mnemonics_list: self.mnemonics_list.insert(0, v); self.ent.delete(0, tk.END); self.refresh()
    def rem(self):
        s = self.lb.curselection()
        if s and messagebox.askyesno("?", "Apagar?"): self.mnemonics_list.pop(s[0]); self.refresh()
    def save(self):
        if self.save_cb(self.full_map): self.destroy()

class ChannelMapperDialog(tk.Toplevel):
    def __init__(self, parent, avail, guess):
        super().__init__(parent)
        self.title("Mapear"); self.geometry("350x300"); self.result = None
        self.sels = {}
        for k in ["PRESSURE","TIME","VOLUME","DEPTH"]:
            f = ttk.Frame(self); f.pack(fill='x', pady=2, padx=5)
            ttk.Label(f, text=k).pack(side='left')
            cb = ttk.Combobox(f, values=avail); cb.pack(side='right', fill='x', expand=True)
            if k in guess: cb.set(guess[k])
            self.sels[k] = cb
        ttk.Button(self, text="OK", command=self.ok).pack(pady=10)
        self.transient(parent); center_window_on_parent(self, parent); self.grab_set()
        parent.wait_window(self)
    def ok(self):
        if not self.sels['PRESSURE'].get(): return
        self.result = {k: [v.get()] for k, v in self.sels.items() if v.get()}
        self.destroy()

# =============================================================================
# 4. APP PRINCIPAL
# =============================================================================
class GeoApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("GeoViewer Pro v94 - Dual Grid Visibility")
        self.geometry("1200x750")
        try: self.state('zoomed') 
        except: pass

        # DADOS
        self.data_store = {} 
        self.logical_files_store = None
        self.current_key = None 
        self.current_well_name = ""

        # CONFIG
        self.active_mnemonic_map = load_library_from_disk()
        self.active_style = copy.deepcopy(DEFAULT_STYLE)

        # ESTADOS
        self.is_analyzing = False 
        self.is_interpreting = False
        self.cid_motion = None; self.cid_click = None

        # INTERPRETAÇÃO
        self.interpretation_store = {} 
        self.interp_sequence = []      
        self.interp_step = 0            
        self.interp_temp_data = {}     
        self.interp_artists = []        

        self.cursor_line_v = None; self.cursor_line_h = None; self.cursor_annot = None
        
        # Move Event
        self.is_moving_event = False
        self.event_to_move = None
        
        self.cursors = []

        self.columnconfigure(1, weight=1); self.rowconfigure(0, weight=1)
        self._setup_ui()
        self._draw_watermark()

    def _setup_ui(self):
        frm_side = ttk.Frame(self, padding=10, relief="groove")
        frm_side.grid(row=0, column=0, sticky="nsew")

        ttk.Label(frm_side, text="Arquivo:", font=("Arial", 9, "bold")).pack(anchor="w")
        ttk.Button(frm_side, text="📂 Carregar DLIS", command=self.open_file).pack(fill="x", pady=2)
        ttk.Button(frm_side, text="❌ Resetar", command=self.reset_workspace).pack(fill="x", pady=2)
        ttk.Separator(frm_side, orient='horizontal').pack(fill='x', pady=10)

        ttk.Label(frm_side, text="Configuração:", font=("Arial", 9, "bold")).pack(anchor="w")
        ttk.Button(frm_side, text="🎨 Estilo", command=self.open_style_editor).pack(fill="x", pady=2)
        ttk.Button(frm_side, text="📚 Biblioteca", command=self.open_library_editor).pack(fill="x", pady=2)
        ttk.Button(frm_side, text="🔄 Remapear Canal", command=self.redefine_channels).pack(fill="x", pady=2)
        ttk.Separator(frm_side, orient='horizontal').pack(fill='x', pady=10)

        ttk.Label(frm_side, text="Interpretação:", font=("Arial", 9, "bold")).pack(anchor="w")
        self.btn_analyze = tk.Button(frm_side, text="🔍 Analisar Valores", command=self.toggle_analysis, relief="raised", bg="#f0f0f0")
        self.btn_analyze.pack(fill="x", pady=2)
        self.btn_interpret = tk.Button(frm_side, text="📍 Interpretar", command=self.toggle_interpretation, relief="raised", bg="#f0f0f0")
        self.btn_interpret.pack(fill="x", pady=2)

        f_interp_ops = ttk.Frame(frm_side)
        f_interp_ops.pack(fill='x', pady=5)
        f_interp_ops.columnconfigure(0, weight=1); f_interp_ops.columnconfigure(1, weight=1); f_interp_ops.columnconfigure(2, weight=1)
        ttk.Button(f_interp_ops, text="👁️ Ver", command=self.view_current_interpretation).grid(row=0, column=0, sticky='ew', padx=1)
        ttk.Button(f_interp_ops, text="🗑️ Limpar", command=self.reset_current_interpretation).grid(row=0, column=1, sticky='ew', padx=1)
        ttk.Button(f_interp_ops, text="💾 Excel", command=self.save_all_interpretations).grid(row=0, column=2, sticky='ew', padx=1)
        ttk.Button(f_interp_ops, text="✏️ Mover", command=self.start_move_event).grid(row=1, column=1, sticky='ew', padx=1)

        ttk.Button(frm_side, text="📤 Export LAS/CSV", command=self.export_current_data).pack(fill="x", pady=5)
        ttk.Separator(frm_side, orient='horizontal').pack(fill='x', pady=10)

        ttk.Label(frm_side, text="Testes:", font=("Arial", 9, "bold")).pack(anchor="w")
        frm_list = ttk.Frame(frm_side); frm_list.pack(fill="both", expand=True, pady=5)
        sb = ttk.Scrollbar(frm_list, orient="vertical")
        self.listbox = tk.Listbox(frm_list, yscrollcommand=sb.set, selectmode=tk.SINGLE, font=("Consolas", 9))
        sb.config(command=self.listbox.yview); sb.pack(side="right", fill="y"); self.listbox.pack(side="left", fill="both", expand=True)
        self.listbox.bind("<<ListboxSelect>>", self.on_select_test)

        self.frm_channels_container = ttk.Frame(frm_side); self.frm_channels_container.pack(fill='x', pady=5)
        self.lbl_info = ttk.Label(frm_side, text="Aguardando...", foreground="#555"); self.lbl_info.pack(pady=5)
        ttk.Button(frm_side, text="🚪 Sair", command=self.quit_app).pack(fill="x", side="bottom")

        frm_main = ttk.Frame(self, relief="flat"); frm_main.grid(row=0, column=1, sticky="nsew")
        
        # --- LAYOUT DIVIDIDO (SPLIT) ---
        self.fig, (self.ax1, self.ax2) = plt.subplots(2, 1, figsize=(8, 6), dpi=100, sharex=True, 
                                                      gridspec_kw={'height_ratios': [3, 1]})
        plt.subplots_adjust(hspace=0.05)
        
        self.canvas = FigureCanvasTkAgg(self.fig, master=frm_main); self.canvas.draw(); self.canvas.get_tk_widget().pack(fill="both", expand=True)
        self.toolbar = NavigationToolbar2Tk(self.canvas, frm_main); self.toolbar.update(); self.canvas.get_tk_widget().pack(fill="x", side="bottom")
        self._connect_events()

    def _connect_events(self):
        if self.cid_motion: self.canvas.mpl_disconnect(self.cid_motion)
        if self.cid_click: self.canvas.mpl_disconnect(self.cid_click)
        self.cid_motion = self.canvas.mpl_connect("motion_notify_event", self.on_mouse_move)
        self.cid_click = self.canvas.mpl_connect("button_press_event", self.on_canvas_click)

    # --- CONTROLE ---
    def open_file(self):
        if self.data_store: self.reset_workspace()
        path = filedialog.askopenfilename(filetypes=[("DLIS", "*.dlis"), ("All", "*.*")])
        if not path: return
        self.config(cursor="watch"); self.lbl_info.config(text="Lendo..."); self.update()
        res, lfs, wname, err = backend_load_dlis_smart(path, self.active_mnemonic_map, self)
        self.config(cursor="")
        if err: messagebox.showerror("Erro", err); return
        self.data_store = res; self.logical_files_store = lfs; self.current_well_name = wname
        for k in self.data_store: self.listbox.insert(tk.END, k)
        self.lbl_info.config(text=f"{len(res)} testes carregados.", foreground="green")

    def reset_workspace(self):
        if self.data_store and not messagebox.askyesno("Confirmar", "Fechar?"): return
        self.data_store = {}; self.logical_files_store = None; self.current_key = None; self.interpretation_store = {}
        self.listbox.delete(0, tk.END); self.lbl_info.config(text="Pronto."); self._draw_watermark()
        for widget in self.frm_channels_container.winfo_children(): widget.destroy()
        self.is_analyzing = False; self.is_interpreting = False; self.is_moving_event = False
        self.btn_analyze.config(relief="raised", bg="#f0f0f0", text="🔍 Analisar Valores")
        self.btn_interpret.config(relief="raised", bg="#f0f0f0", text="📍 Interpretar")
        for c in self.cursors: 
            try: c.remove()
            except: pass
        self.cursors = []

    def quit_app(self):
        if messagebox.askokcancel("Sair", "Sair?"): self.destroy()

    def redefine_channels(self):
        if not self.logical_files_store: messagebox.showwarning("Aviso", "Carregue um arquivo."); return
        all_c = get_all_channels_from_dlis(self.logical_files_store)
        dial = ChannelMapperDialog(self, all_c, guess_channels(all_c, self.active_mnemonic_map))
        if dial.result:
            self.config(cursor="watch"); self.update()
            res = extract_data_with_map(self.logical_files_store, dial.result)
            self.config(cursor="")
            if res:
                self.data_store = res; self.listbox.delete(0, tk.END)
                for k in self.data_store: self.listbox.insert(tk.END, k)
                self._draw_watermark(); self.lbl_info.config(text=f"{len(res)} testes remapeados.")

    def on_select_test(self, event):
        sel = self.listbox.curselection()
        if not sel: return
        self.current_key = self.listbox.get(sel[0])
        df = self.data_store[self.current_key]
        uc = df.attrs.get('channels_used', {})
        for widget in self.frm_channels_container.winfo_children(): widget.destroy()
        p_val = uc.get('PRESSURE', 'N/A'); v_val = uc.get('VOLUME', 'N/A')
        t_val = uc.get('TIME', 'Index'); d_val = uc.get('DEPTH', 'Calc/Index')
        lbl_p = ttk.Label(self.frm_channels_container, text=f"🔴 P: {p_val}", foreground="blue", font=("Arial", 8))
        lbl_v = ttk.Label(self.frm_channels_container, text=f"💧 V: {v_val}", foreground="blue", font=("Arial", 8))
        lbl_d = ttk.Label(self.frm_channels_container, text=f"📏 D: {d_val}", foreground="blue", font=("Arial", 8))
        lbl_t = ttk.Label(self.frm_channels_container, text=f"⏱️ T: {t_val}", foreground="blue", font=("Arial", 8))
        lbl_p.grid(row=0, column=0, sticky='w', padx=2, pady=1); lbl_v.grid(row=1, column=0, sticky='w', padx=2, pady=1)
        lbl_d.grid(row=0, column=1, sticky='w', padx=5, pady=1); lbl_t.grid(row=1, column=1, sticky='w', padx=5, pady=1)
        if self.is_interpreting:
            self.is_interpreting = False; self.btn_interpret.config(relief="raised", bg="#f0f0f0", text="📍 Interpretar")
            self.lbl_info.config(text="Interpretação pausada.", foreground="orange")
        if self.current_key in self.interpretation_store: self.btn_interpret.config(bg="#90EE90", text="✓ Interpretado")
        else: self.btn_interpret.config(bg="#f0f0f0", text="📍 Interpretar")
        self.plot_data(df, self.current_key)

    def open_library_editor(self):
        CategorySelectorWindow(self, self.active_mnemonic_map, lambda m: [setattr(self, 'active_mnemonic_map', m), save_json(CONFIG_LIB_FILENAME, m)][1])
    def open_style_editor(self):
        StyleEditorWindow(self, self.active_style, lambda s: [setattr(self, 'active_style', s), self.plot_data(self.data_store[self.current_key], self.current_key) if self.current_key else None][1])
    def export_current_data(self):
        if self.current_key: ExportOptionWindow(self)
        else: messagebox.showwarning("Aviso", "Selecione um teste.")

    def perform_export(self, choice, version):
        df = self.data_store[self.current_key]
        ext = ".csv" if choice == "CSV" else ".las"
        path = filedialog.asksaveasfilename(defaultextension=ext, filetypes=[(choice, f"*{ext}")])
        if not path: return
        try:
            if choice == "CSV": df.to_csv(path, index=False)
            else: self._write_las(df, path, self.current_key, version)
            messagebox.showinfo("Sucesso", "Exportado.")
        except Exception as e: messagebox.showerror("Erro", str(e))

    def _write_las(self, df, path, name, version="2.0"):
        df_las = df.copy().fillna(-999.25)
        v_str = "2.0 : CWLS LAS 2.0" if version == "2.0" else "3.0 : CWLS LAS 3.0"
        with open(path, 'w') as f:
            f.write(f"~VERSION INFORMATION\n VERS. {v_str}\n WRAP. NO\n~WELL INFORMATION\n WELL. {self.current_well_name}\n NULL. -999.25\n~CURVE INFORMATION\n")
            for c in df_las.columns: f.write(f" {c:<10}.UNIT : {c}\n")
            f.write("~ASCII\n")
            np.savetxt(f, df_las.values, fmt='%10.4f')

    def _draw_watermark(self):
        # Limpa e prepara ambos os eixos com o estilo padrão, mas vazios
        self.ax1.clear(); self.ax2.clear()
        st = self.active_style
        
        # Aplica estilo no AX1 (Pressão)
        self.ax1.set_facecolor(st.get('plot_bgcolor', 'white'))
        if st['show_grid']:
             self.ax1.grid(True, linestyle='--', alpha=st['grid_alpha'], color=st['grid_color'], linewidth=st['grid_linewidth'])
        
        # Aplica estilo no AX2 (Volume)
        self.ax2.set_facecolor(st.get('vol_plot_bgcolor', 'white'))
        if st['show_grid']:
             self.ax2.grid(True, linestyle='--', alpha=st['grid_alpha'], color=st['grid_color'], linewidth=st['grid_linewidth'])

        # Esconde Ticks apenas para o modo watermark
        self.ax1.set_xticklabels([]); self.ax1.set_yticklabels([])
        self.ax2.set_xticklabels([]); self.ax2.set_yticklabels([])

        # Texto central
        self.fig.text(0.5, 0.5, "GEOVIEWER", fontsize=60, color='gray', alpha=0.1, ha='center', va='center', weight='bold')
        self.canvas.draw()

    def _get_plot_arrays(self, key):
        df = self.data_store[key]; st = self.active_style
        if 'TIME' in df.columns:
            raw_t = df['TIME'].values
            x = (raw_t - raw_t.min()) / 1000.0 if raw_t.max() > 10000 else raw_t
        else: x = np.array(df.index)
        x_base = x.copy()
        if st.get('use_time_conv', False): 
            factor = st.get('time_conv_factor', 1.0)
            if factor == 0: factor = 1.0
            x_visual = x_base / factor
        else: x_visual = x_base
        use_smooth = ('PRE_SMOOTH' in df.columns) and st.get('show_smooth', True)
        y_source_psi = df['PRE_SMOOTH'].values if use_smooth else df['PRESSURE'].values
        pres_factor = st.get('conversion_factor', 1.0) if st.get('convert_psi_kgf', False) else 1.0
        if pres_factor == 0: pres_factor = 1.0
        return x_visual, y_source_psi / pres_factor, y_source_psi, df, x_base

    def start_move_event(self):
        if not self.current_key: messagebox.showwarning("Aviso", "Selecione um teste."); return
        if self.current_key not in self.interpretation_store: messagebox.showinfo("Info", "Sem dados."); return
        def on_selected(evt_name):
            self.event_to_move = evt_name; self.is_moving_event = True; self.is_interpreting = False 
            self.lbl_info.config(text=f"EDITANDO: {evt_name}", foreground="orange", font=("Arial", 10, "bold"))
            self.canvas.get_tk_widget().focus_force()
            if self.toolbar.mode != '': self.toolbar.mode = ''; self.toolbar.update()
            if not self.is_analyzing: self.toggle_analysis()
        SelectEventDialog(self, list(self.interpretation_store[self.current_key].keys()), on_selected)

    def reset_current_interpretation(self):
        if not self.current_key: return
        for artist in self.interp_artists: 
            try: artist.remove()
            except: pass
        self.interp_artists = []
        if self.current_key in self.interpretation_store: del self.interpretation_store[self.current_key]
        self.is_interpreting = False; self.is_moving_event = False
        self.interp_step = 0; self.interp_temp_data = {}
        self.btn_interpret.config(relief="raised", bg="#f0f0f0", text="📍 Interpretar")
        self.lbl_info.config(text="Interpretação limpa.", foreground="black")
        self.canvas.draw_idle()

    def toggle_interpretation(self):
        if not self.current_key: messagebox.showwarning("Aviso", "Selecione um teste."); return
        if self.is_interpreting:
            self.is_interpreting = False; self.btn_interpret.config(relief="raised", bg="#f0f0f0", text="📍 Interpretar")
            self.lbl_info.config(text="Cancelado.", foreground="black"); self.toolbar.mode = ''; self.toolbar.update()
            if self.current_key not in self.interpretation_store:
                for a in self.interp_artists: 
                    try: a.remove() 
                    except: pass
                self.interp_artists = []
                self.canvas.draw_idle()
            if not self.is_analyzing:
                for c in self.cursors: c.set_visible(False)
                self.cursor_annot.set_visible(False)
                self.canvas.draw_idle()
            return
        n = simpledialog.askinteger("Ciclos", "Quantos pares Fluxo/Estática?", initialvalue=1, minvalue=1, maxvalue=10, parent=self)
        if not n: return
        self.interp_sequence = ['PHI']
        for i in range(1, n + 1): self.interp_sequence.extend([f'PFI-{i}', f'PFF-{i}', f'PEI-{i}', f'PEF-{i}'])
        self.interp_sequence.append('PHF')
        self.interp_step = 0; self.interp_temp_data = {}
        for a in self.interp_artists: 
            try: a.remove()
            except: pass
        self.interp_artists = []
        self.is_interpreting = True; self.is_moving_event = False
        self.canvas.get_tk_widget().focus_force()
        if self.toolbar.mode != '':
             if 'ZOOM' in self.toolbar.mode.upper(): self.toolbar.zoom()
             elif 'PAN' in self.toolbar.mode.upper(): self.toolbar.pan()
        if not self.is_analyzing: self.toggle_analysis()
        self.btn_interpret.config(relief="sunken", bg="#FFD700", text="📍 INTERPRETANDO...")
        self.lbl_info.config(text=f"👉 CLIQUE NO PONTO: {self.interp_sequence[0]}", foreground="red", font=("Arial", 10, "bold"))
        messagebox.showinfo("Modo Interpretação", f"Sequência: {' > '.join(self.interp_sequence)}\n\nClique no gráfico.")

    def on_canvas_click(self, event):
        if not (self.is_interpreting or self.is_moving_event): return
        if not self.current_key: return
        if event.button != 1: return
        if self.toolbar.mode != "": return
        # ALLOW CROSS-PLOT CLICK
        if event.inaxes not in [self.ax1, self.ax2]: return 
        click_x_data = event.xdata
        if click_x_data is None:
            bbox = self.ax1.get_window_extent()
            if bbox.contains(event.x, event.y):
                inv_transform = self.ax1.transData.inverted(); data_coords = inv_transform.transform((event.x, event.y)); click_x_data = data_coords[0]
            else: return
        try:
            x_plot, y_plot, y_psi_source, df, x_base = self._get_plot_arrays(self.current_key)
            idx = np.argmin(np.abs(x_plot - click_x_data))
            saved_time_base = float(x_base[idx]); saved_pres_psi = float(y_psi_source[idx])
            real_time_vis = float(x_plot[idx]); real_pres_vis = float(y_plot[idx])

            if self.is_moving_event and self.event_to_move:
                self.interpretation_store[self.current_key][self.event_to_move] = {'time_base': saved_time_base, 'pres_raw_psi': saved_pres_psi}
                print(f"✓ Movido: {self.event_to_move}"); self.is_moving_event = False; self.event_to_move = None
                self.lbl_info.config(text="Evento atualizado.", foreground="green")
                self.plot_data(df, self.current_key); return

            evt = self.interp_sequence[self.interp_step]
            self.interp_temp_data[evt] = {'time_base': saved_time_base, 'pres_raw_psi': saved_pres_psi}
            color = 'blue' if ('PHI' in evt or 'PHF' in evt) else ('green' if ('PFI' in evt or 'PFF' in evt) else 'red')
            marker, = self.ax1.plot(real_time_vis, real_pres_vis, marker='o', color=color, markersize=10, markeredgecolor='white', markeredgewidth=2, zorder=100)
            text = self.ax1.text(real_time_vis, real_pres_vis, f"  {evt}\n  {saved_pres_psi:.1f}", color=color, fontweight='bold', fontsize=8, verticalalignment='bottom', horizontalalignment='left', zorder=101, bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.8))
            self.interp_artists.extend([marker, text]); self.canvas.draw_idle()
            print(f"✓ Registrado: {evt}")
            self.interp_step += 1
            if self.interp_step >= len(self.interp_sequence):
                self.interpretation_store[self.current_key] = copy.deepcopy(self.interp_temp_data)
                messagebox.showinfo("Fim", f"Salvo com sucesso!")
                self.is_interpreting = False; self.btn_interpret.config(relief="raised", bg="#90EE90", text="✓ Interpretado")
                self.lbl_info.config(text=f"Salvo.", foreground="green"); self.toolbar.mode = ''; self.toolbar.update()
            else:
                next_evt = self.interp_sequence[self.interp_step]
                self.lbl_info.config(text=f"👉 CLIQUE NO PONTO: {next_evt}")
        except Exception as e: print(f"❌ Erro: {e}"); traceback.print_exc()

    def view_current_interpretation(self):
        if not self.current_key: return
        if self.current_key not in self.interpretation_store: messagebox.showinfo("Info", "Sem dados."); return
        data = self.interpretation_store[self.current_key]
        top = tk.Toplevel(self); top.title(f"Interpretação: {self.current_key}"); top.geometry("500x600")
        frm_table = ttk.Frame(top, padding=10); frm_table.pack(fill='both', expand=True)
        txt = tk.Text(frm_table, font=("Consolas", 9), wrap='none'); txt.pack(side='left', fill='both', expand=True)
        header = f"{'EVENTO':<12} | {'PSI':>15} | {'TEMPO(Base)':>12}\n"; txt.insert(tk.END, header + "-"*45 + "\n")
        for e, v in data.items(): 
            t = v.get('time_base', v.get('time_plot', 0))
            txt.insert(tk.END, f"{e:<12} | {v['pres_raw_psi']:>15.2f} | {t:>12.2f}\n")
        txt.config(state='disabled'); center_window_on_parent(top, self)

    def save_all_interpretations(self):
        if not self.interpretation_store: return
        path = filedialog.asksaveasfilename(defaultextension=".xlsx", filetypes=[("Excel", "*.xlsx"), ("CSV", "*.csv")])
        if not path: return
        try:
            rows = []
            for test_name, eventos in self.interpretation_store.items():
                row = {'POCO': self.current_well_name, 'TESTE': test_name}
                for e, v in eventos.items(): 
                    t = v.get('time_base', v.get('time_plot', 0))
                    row[f"{e}_PSI"] = v['pres_raw_psi']; row[f"{e}_TEMPO"] = t
                rows.append(row)
            df = pd.DataFrame(rows)
            if path.endswith('.csv'): df.to_csv(path, index=False)
            else: df.to_excel(path, index=False)
            messagebox.showinfo("Sucesso", "Exportado.")
        except Exception as e: messagebox.showerror("Erro", str(e))

    def toggle_analysis(self):
        if (self.is_interpreting or self.is_moving_event) and self.is_analyzing: return
        self.is_analyzing = not self.is_analyzing
        if self.is_analyzing:
            self.btn_analyze.config(relief="sunken", bg="#98FB98", text="🔍 Analisar: ON")
        else:
            self.btn_analyze.config(relief="raised", bg="#f0f0f0", text="🔍 Analisar Valores")
            if self.cursor_line_v: self.cursor_line_v.set_visible(False); self.cursor_line_h.set_visible(False); self.cursor_annot.set_visible(False); self.canvas.draw_idle()

    def on_mouse_move(self, event):
        if not self.is_analyzing or not self.current_key or not self.cursors: return
        if event.inaxes in [self.ax1, self.ax2]:
            x_plot, y_plot, _, df, _ = self._get_plot_arrays(self.current_key)
            idx = np.argmin(np.abs(x_plot - event.xdata))
            t_val = x_plot[idx]; p_val = y_plot[idx]
            st = self.active_style; u = st.get('conversion_label', 'kgf/cm²') if st.get('convert_psi_kgf', False) else "psi"
            txt = f"T: {t_val:.1f}\nP: {p_val:.1f} {u}"
            vol_val = 0.0
            if 'VOLUME' in df.columns:
                v_idx = idx if len(df) == len(x_plot) else int((len(df)-1) * (idx / max(1, len(x_plot))))
                vol_val = df['VOLUME'].iloc[v_idx]
                txt += f"\nV: {vol_val:.1f}"
            if self.is_interpreting: ne = self.interp_sequence[self.interp_step] if self.interp_step < len(self.interp_sequence) else "FIM"; txt += f"\n[ALVO: {ne}]"
            if self.is_moving_event and self.event_to_move: txt += f"\n[EDITANDO: {self.event_to_move}]"

            self.cursors[0].set_xdata([t_val, t_val])
            self.cursors[1].set_xdata([t_val, t_val])
            
            if event.inaxes == self.ax1:
                 self.cursors[2].set_visible(True); self.cursors[2].set_ydata([p_val, p_val])
                 self.cursors[3].set_visible(False)
                 self.cursor_annot.xy = (t_val, p_val)
            elif event.inaxes == self.ax2:
                 self.cursors[2].set_visible(False)
                 self.cursors[3].set_visible(True); self.cursors[3].set_ydata([vol_val, vol_val])
                 self.cursor_annot.xy = (t_val, vol_val)

            self.cursor_annot.set_text(txt)
            self.cursors[0].set_visible(True); self.cursors[1].set_visible(True)
            self.cursor_annot.set_visible(True)
            self.canvas.draw_idle()
        else:
            if self.cursors and self.cursors[0].get_visible():
                for c in self.cursors: c.set_visible(False)
                self.cursor_annot.set_visible(False)
                self.canvas.draw_idle()

    def plot_data(self, df, title):
        self.ax1.clear(); self.ax2.clear(); st = self.active_style
        x_vis, y_vis, _, _, _ = self._get_plot_arrays(title)
        try:
            tx = float(st.get('tick_step_x', 0))
            if tx > 0: self.ax1.xaxis.set_major_locator(mticker.MultipleLocator(tx))
            else: self.ax1.xaxis.set_major_locator(mticker.AutoLocator())
        except: self.ax1.xaxis.set_major_locator(mticker.AutoLocator())
        try:
            ty = float(st.get('tick_step_y', 0))
            if ty > 0: self.ax1.yaxis.set_major_locator(mticker.MultipleLocator(ty))
            else: self.ax1.yaxis.set_major_locator(mticker.AutoLocator())
        except: self.ax1.yaxis.set_major_locator(mticker.AutoLocator())
        
        base_label = st['label_y_left']
        if st.get('convert_psi_kgf', False):
            target_unit = st.get('conversion_label', 'kgf/cm²')
            if 'psi' in base_label.lower(): label_y = re.sub(r'psi', target_unit, base_label, flags=re.IGNORECASE)
            else: label_y = f"{base_label} ({target_unit})"
        else: label_y = base_label
            
        self.ax1.set_xlabel(st['label_x'], fontsize=st.get('fs_label_x', 10))
        self.ax1.set_ylabel(label_y, fontweight='bold', fontsize=st.get('fs_label_y', 10))
        self.ax1.set_facecolor(st.get('plot_bgcolor', 'white'))
        if st.get('show_grid', True): self.ax1.grid(True, linestyle='--', alpha=st.get('grid_alpha', 0.5), color=st.get('grid_color', '#b0b0b0'), linewidth=st.get('grid_linewidth', 0.8))
        if st.get('invert_y_left', False): self.ax1.invert_yaxis()
        pres_factor = st.get('conversion_factor', 1.0) if st.get('convert_psi_kgf', False) else 1.0
        if pres_factor == 0: pres_factor = 1.0
        y_raw_bg = df['PRESSURE'].values / pres_factor
        
        lns = []
        l1, = self.ax1.plot(x_vis, y_raw_bg, color=st['col_pres_raw'], alpha=st.get('alpha_pres_raw', 0.3), lw=st.get('lw_pres_raw', 1), label='Bruta', zorder=1)
        lns.append(l1)
        if ('PRE_SMOOTH' in df.columns) and st.get('show_smooth', True): 
            l2, = self.ax1.plot(x_vis, y_vis, color=st['col_pres_smooth'], lw=st.get('lw_pres_smooth', 1.5), label='Suavizada', zorder=2)
            lns.append(l2)
            
        # SMART SCALE (v90)
        valid_y = y_raw_bg[~np.isnan(y_raw_bg)]
        if len(valid_y) > 0:
            p1, p99 = np.percentile(valid_y, [0.1, 99.9])
            span = p99 - p1; margin = span * 0.1
            if span > 0: self.ax1.set_ylim(p1 - margin, p99 + margin)

        # AX2 VOLUME
        self.ax2.set_xlabel(st['label_x'], fontsize=st.get('fs_label_x', 10))
        self.ax2.set_ylabel(st['label_y_right'], fontweight='bold', fontsize=st.get('fs_label_y', 10))
        self.ax2.yaxis.set_label_position("left") 
        self.ax2.tick_params(axis='y', labelleft=True, labelright=False)
        self.ax2.set_facecolor(st.get('vol_plot_bgcolor', 'white')) # COR FUNDO VOL
        
        # GRID VOLUME
        if st.get('show_grid', True): self.ax2.grid(True, linestyle='--', alpha=st.get('grid_alpha', 0.5), color=st.get('grid_color', '#b0b0b0'), linewidth=st.get('grid_linewidth', 0.8))
        
        if 'VOLUME' in df.columns:
            if st.get('invert_y_right', False): self.ax2.invert_yaxis()
            l3, = self.ax2.plot(x_vis, df['VOLUME'], color=st['col_vol'], lw=st.get('lw_vol', 1.5), label='Volume', zorder=2)
            lns.append(l3)
            try:
                ty_r = float(st.get('tick_step_y_right', 0))
                if ty_r > 0: self.ax2.yaxis.set_major_locator(mticker.MultipleLocator(ty_r))
                else: self.ax2.yaxis.set_major_locator(mticker.AutoLocator())
            except: self.ax2.yaxis.set_major_locator(mticker.AutoLocator())
        else: self.ax2.axis('off')
        
        self.ax1.set_xlim(x_vis.min(), x_vis.max()); self.ax1.set_title(f"{st['title_prefix']} {self.current_well_name} | {title}", fontweight='bold')
        
        self.ax1.legend(lns, [l.get_label() for l in lns], loc=st.get('legend_loc', 'upper right'), frameon=True, facecolor='white', framealpha=0.9)

        self.interp_artists = []
        if title in self.interpretation_store:
            for evento, valores in self.interpretation_store[title].items():
                color = 'blue' if ('PHI' in evento or 'PHF' in evento) else ('green' if ('PFI' in evento or 'PFF' in evento) else 'red')
                t_base = valores.get('time_base', valores.get('time_plot', 0))
                t_factor = st.get('time_conv_factor', 1.0) if st.get('use_time_conv', False) else 1.0
                if t_factor == 0: t_factor = 1.0
                t_vis = t_base / t_factor
                p_vis = valores['pres_raw_psi'] / pres_factor
                mk, = self.ax1.plot(t_vis, p_vis, marker='o', color=color, markersize=10, markeredgecolor='white', zorder=100)
                tx = self.ax1.text(t_vis, p_vis, f"  {evento}", color=color, fontweight='bold', fontsize=8, verticalalignment='bottom', zorder=101)
                self.interp_artists.extend([mk, tx])
        
        # CURSORES
        self.cursors = []
        self.cursors.append(self.ax1.axvline(x=0, color='gray', ls=':', lw=1, zorder=50, visible=False))
        self.cursors.append(self.ax2.axvline(x=0, color='gray', ls=':', lw=1, zorder=50, visible=False))
        self.cursors.append(self.ax1.axhline(y=0, color='gray', ls=':', lw=1, zorder=50, visible=False))
        self.cursors.append(self.ax2.axhline(y=0, color='gray', ls=':', lw=1, zorder=50, visible=False))
        self.cursor_annot = self.ax1.annotate("", xy=(0, 0), xytext=(10, 10), textcoords="offset points", bbox=dict(boxstyle="round", fc="yellow", alpha=0.9), fontsize=9, zorder=102, visible=False)
        self.fig.tight_layout(); self.canvas.draw()

if __name__ == "__main__":
    app = GeoApp()
    app.mainloop()

✓ Registrado: PHI
✓ Registrado: PFI-1
✓ Registrado: PFF-1
✓ Registrado: PEI-1
✓ Registrado: PEF-1
✓ Registrado: PFI-2
✓ Registrado: PFF-2
✓ Registrado: PEI-2
✓ Registrado: PEF-2
✓ Registrado: PHF
✓ Movido: PEF-2
✓ Registrado: PHI
✓ Registrado: PFI-1
✓ Registrado: PFF-1
✓ Registrado: PEI-1
✓ Registrado: PEF-1
✓ Registrado: PFI-2
✓ Registrado: PFF-2
✓ Registrado: PEI-2
✓ Registrado: PEF-2
✓ Registrado: PHF
✓ Movido: PHF
✓ Movido: PHF
